In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Copy all datasets from Drive
!cp /content/drive/MyDrive/milestone2_data/*.csv .
!ls *.csv

Mounted at /content/drive
Iris.csv		shuttle.csv	  synthetic_1000k_200f.csv
letter-recognition.csv	Skin_NonSkin.csv


In [2]:
from google.colab import files
print("Upload milestone2_code.cu:")
uploaded = files.upload()

Upload milestone2_code.cu:


Saving milestone2_code.cu to milestone2_code.cu


In [4]:
# Read the file, add validation, write back
with open('milestone2_code.cu', 'r') as f:
    content = f.read()

# Find where to insert the validation function (after count_unique_classes)
validation_func = '''
static bool validate_dataset(const Dataset& data, const std::string& name) {
    if (data.features.empty() || data.labels.empty()) {
        std::cerr << "[!] Dataset '" << name << "' is empty.\\n";
        return false;
    }
    if (data.features.size() != data.labels.size()) {
        std::cerr << "[!] Dataset '" << name << "' size mismatch.\\n";
        return false;
    }
    return true;
}
'''

# Insert after count_unique_classes function ends
content = content.replace(
    'static int count_unique_classes(const std::vector<int>& labels) {',
    validation_func + '\nstatic int count_unique_classes(const std::vector<int>& labels) {'
)

# Add validation call in main loop
content = content.replace(
    'data = load_csv(path, spec.has_header);',
    'data = load_csv(path, spec.has_header);\n            if (!validate_dataset(data, spec.name)) { continue; }'
)

with open('milestone2_code.cu', 'w') as f:
    f.write(content)

print("✅ Fixed code with validation")

✅ Fixed code with validation


In [5]:
!nvcc -O2 -std=c++17 -arch=sm_75 milestone2_code.cu -o milestone2
!./milestone2

   MILESTONE 2 — Final Integrated CPU-GPU Tree Trainer

[Test: pure node -> leaf]
  [PASS]

[Test: level-wise toy split]
  [PASS]

[Loading] Iris from: Iris.csv

[Dataset: Iris | Mode: GPU (Hybrid CPU-GPU)]
  samples/features/classes : 150 / 5 / 3
  train/test samples       : 120 / 30
  total_train_sec          : 0.000513
  data_prep_sec            : 0.000001  (Step 2: index packing)
  split_eval_sec           : 0.000294  (Step 3: GPU kernels — compare vs M1)
  apply_split_sec          : 0.000006  (Step 4: CPU apply splits)
  predict_sec              : 0.000002
  train_acc                : 100.000000%
  test_acc                 : 100.000000%


In [7]:
%%writefile test_load.cpp
#include <iostream>
#include <fstream>
#include <sstream>
#include <vector>
#include <string>

int main() {
    std::ifstream file("shuttle.csv");
    if (!file.is_open()) {
        std::cout << "Cannot open file\n";
        return 1;
    }

    std::string line;
    int line_num = 0;
    while (std::getline(file, line)) {
        line_num++;
        if (line_num == 1) continue;

        std::stringstream ss(line);
        std::string token;
        std::vector<std::string> tokens;
        while (std::getline(ss, token, ',')) {
            tokens.push_back(token);
        }

        std::cout << "Line " << line_num << ": " << tokens.size() << " tokens\n";
        if (line_num > 5) break;
    }
    return 0;
}

Writing test_load.cpp


In [8]:
!g++ test_load.cpp -o test_load && ./test_load

Line 2: 10 tokens
Line 3: 10 tokens
Line 4: 10 tokens
Line 5: 10 tokens
Line 6: 10 tokens


In [9]:
!rm -f milestone2_code.cu

from google.colab import files
print("Upload the clean .cu file:")
uploaded = files.upload()

Upload the clean .cu file:


Saving milestone2_code.cu to milestone2_code.cu


In [10]:
# Copy and modify for Iris only
!cp milestone2_code.cu milestone2_iris.cu
!sed -i 's|{"Iris", "Iris.csv", true},|{"Iris", "Iris.csv", true}|; s|{"Shuttle.*|// removed|; s|{"LetterRecognition.*|// removed|; s|{"Skin_NonSkin.*|// removed|; s|{"Synthetic_dataset.*|// removed|' milestone2_iris.cu

# Compile and run Iris
!nvcc -O2 -std=c++17 -arch=sm_75 milestone2_iris.cu -o milestone2_iris
!./milestone2_iris

# Download result
from google.colab import files
files.download('benchmark_metrics_m2.csv')

   MILESTONE 2 — Final Integrated CPU-GPU Tree Trainer

[Test: pure node -> leaf]
  [PASS]

[Test: level-wise toy split]
  [PASS]

[Loading] Iris from: Iris.csv

[Dataset: Iris | Mode: GPU (Hybrid CPU-GPU)]
  samples/features/classes : 150 / 5 / 3
  train/test samples       : 120 / 30
  total_train_sec          : 0.000420
  data_prep_sec            : 0.000001  (Step 2: index packing)
  split_eval_sec           : 0.000260  (Step 3: GPU kernels — compare vs M1)
  apply_split_sec          : 0.000004  (Step 4: CPU apply splits)
  predict_sec              : 0.000002
  train_acc                : 100.000000%
  test_acc                 : 100.000000%


FileNotFoundError: Cannot find file: benchmark_metrics_m2.csv

In [11]:
!cp milestone2_code.cu milestone2_shuttle.cu
!sed -i 's|{"Shuttle", "shuttle.csv", true},|{"Shuttle", "shuttle.csv", true}|; s|{"Iris.*// removed|; s|{"LetterRecognition.*// removed|; s|{"Skin_NonSkin.*// removed|; s|{"Synthetic_dataset.*// removed|' milestone2_shuttle.cu

!nvcc -O2 -std=c++17 -arch=sm_75 milestone2_shuttle.cu -o milestone2_shuttle
!./milestone2_shuttle

sed: -e expression #1, char 97: unknown option to `s'
   MILESTONE 2 — Final Integrated CPU-GPU Tree Trainer

[Test: pure node -> leaf]
  [PASS]

[Test: level-wise toy split]
  [PASS]

[Loading] Iris from: Iris.csv

[Dataset: Iris | Mode: GPU (Hybrid CPU-GPU)]
  samples/features/classes : 150 / 5 / 3
  train/test samples       : 120 / 30
  total_train_sec          : 0.000515
  data_prep_sec            : 0.000001  (Step 2: index packing)
  split_eval_sec           : 0.000294  (Step 3: GPU kernels — compare vs M1)
  apply_split_sec          : 0.000006  (Step 4: CPU apply splits)
  predict_sec              : 0.000002
  train_acc                : 100.000000%
  test_acc                 : 100.000000%


In [12]:
# Create Iris-only with immediate save
!cp milestone2_code.cu milestone2_iris_fixed.cu

# Add save right after printing results
!sed -i '/print_result_summary(res);/a \            save_metrics_csv("benchmark_metrics_m2.csv", metrics);\n            save_scalability_csv("benchmark_scalability_m2.csv", scalability_rows);' milestone2_iris_fixed.cu

# Compile and run
!nvcc -O2 -std=c++17 -arch=sm_75 milestone2_iris_fixed.cu -o milestone2_iris_fixed
!./milestone2_iris_fixed

# Download
from google.colab import files
files.download('benchmark_metrics_m2.csv')

   MILESTONE 2 — Final Integrated CPU-GPU Tree Trainer

[Test: pure node -> leaf]
  [PASS]

[Test: level-wise toy split]
  [PASS]

[Loading] Iris from: Iris.csv

[Dataset: Iris | Mode: GPU (Hybrid CPU-GPU)]
  samples/features/classes : 150 / 5 / 3
  train/test samples       : 120 / 30
  total_train_sec          : 0.000405
  data_prep_sec            : 0.000001  (Step 2: index packing)
  split_eval_sec           : 0.000224  (Step 3: GPU kernels — compare vs M1)
  apply_split_sec          : 0.000004  (Step 4: CPU apply splits)
  predict_sec              : 0.000002
  train_acc                : 100.000000%
  test_acc                 : 100.000000%


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
# Create Shuttle-only with immediate save
!cp milestone2_code.cu milestone2_shuttle_fixed.cu

# Keep only Shuttle in dataset list
!sed -i '/std::vector<DatasetSpec> datasets = {/,/};/c\        std::vector<DatasetSpec> datasets = {\n            {"Shuttle", "shuttle.csv", true}\n        };' milestone2_shuttle_fixed.cu

# Add save right after printing results
!sed -i '/print_result_summary(res);/a \            save_metrics_csv("benchmark_metrics_m2.csv", metrics);\n            save_scalability_csv("benchmark_scalability_m2.csv", scalability_rows);' milestone2_shuttle_fixed.cu

# Compile
!nvcc -O2 -std=c++17 -arch=sm_75 milestone2_shuttle_fixed.cu -o milestone2_shuttle_fixed

# Run
!./milestone2_shuttle_fixed

   MILESTONE 2 — Final Integrated CPU-GPU Tree Trainer

[Test: pure node -> leaf]
  [PASS]

[Test: level-wise toy split]
  [PASS]

[Loading] Shuttle from: shuttle.csv

[Dataset: Shuttle | Mode: GPU (Hybrid CPU-GPU)]
  samples/features/classes : 14500 / 9 / 7
  train/test samples       : 11600 / 2900
  total_train_sec          : 0.005504
  data_prep_sec            : 0.000064  (Step 2: index packing)
  split_eval_sec           : 0.001822  (Step 3: GPU kernels — compare vs M1)
  apply_split_sec          : 0.001039  (Step 4: CPU apply splits)
  predict_sec              : 0.000202
  train_acc                : 97.775862%
  test_acc                 : 98.206897%

[Artifacts saved]
  - benchmark_metrics_m2.csv
  - benchmark_scalability_m2.csv

Done. Compare split_eval_sec here vs M1 CSV for speedup figures.


In [14]:
# Create Letter-only with immediate save
!cp milestone2_code.cu milestone2_letter_fixed.cu

# Keep only Letter
!sed -i '/std::vector<DatasetSpec> datasets = {/,/};/c\        std::vector<DatasetSpec> datasets = {\n            {"LetterRecognition", "letter-recognition.csv", true}\n        };' milestone2_letter_fixed.cu

# Add save
!sed -i '/print_result_summary(res);/a \            save_metrics_csv("benchmark_metrics_m2.csv", metrics);\n            save_scalability_csv("benchmark_scalability_m2.csv", scalability_rows);' milestone2_letter_fixed.cu

# Compile and run
!nvcc -O2 -std=c++17 -arch=sm_75 milestone2_letter_fixed.cu -o milestone2_letter_fixed
!./milestone2_letter_fixed

   MILESTONE 2 — Final Integrated CPU-GPU Tree Trainer

[Test: pure node -> leaf]
  [PASS]

[Test: level-wise toy split]
  [PASS]

[Loading] LetterRecognition from: letter-recognition.csv

[Dataset: LetterRecognition | Mode: GPU (Hybrid CPU-GPU)]
  samples/features/classes : 20000 / 16 / 26
  train/test samples       : 16000 / 4000
  total_train_sec          : 0.013309
  data_prep_sec            : 0.000094  (Step 2: index packing)
  split_eval_sec           : 0.002825  (Step 3: GPU kernels — compare vs M1)
  apply_split_sec          : 0.001666  (Step 4: CPU apply splits)
  predict_sec              : 0.000352
  train_acc                : 60.112500%
  test_acc                 : 58.575000%

[Artifacts saved]
  - benchmark_metrics_m2.csv
  - benchmark_scalability_m2.csv

Done. Compare split_eval_sec here vs M1 CSV for speedup figures.


In [15]:
# Create Skin-only
!cp milestone2_code.cu milestone2_skin_fixed.cu

# Keep only Skin
!sed -i '/std::vector<DatasetSpec> datasets = {/,/};/c\        std::vector<DatasetSpec> datasets = {\n            {"Skin_NonSkin", "Skin_NonSkin.csv", true}\n        };' milestone2_skin_fixed.cu

# Add save
!sed -i '/print_result_summary(res);/a \            save_metrics_csv("benchmark_metrics_m2.csv", metrics);\n            save_scalability_csv("benchmark_scalability_m2.csv", scalability_rows);' milestone2_skin_fixed.cu

# Compile and run
!nvcc -O2 -std=c++17 -arch=sm_75 milestone2_skin_fixed.cu -o milestone2_skin_fixed
!./milestone2_skin_fixed

   MILESTONE 2 — Final Integrated CPU-GPU Tree Trainer

[Test: pure node -> leaf]
  [PASS]

[Test: level-wise toy split]
  [PASS]

[Loading] Skin_NonSkin from: Skin_NonSkin.csv

[Dataset: Skin_NonSkin | Mode: GPU (Hybrid CPU-GPU)]
  samples/features/classes : 245057 / 3 / 2
  train/test samples       : 196045 / 49012
  total_train_sec          : 0.040398
  data_prep_sec            : 0.000789  (Step 2: index packing)
  split_eval_sec           : 0.005438  (Step 3: GPU kernels — compare vs M1)
  apply_split_sec          : 0.015838  (Step 4: CPU apply splits)
  predict_sec              : 0.002967
  train_acc                : 99.216506%
  test_acc                 : 99.277728%

[Artifacts saved]
  - benchmark_metrics_m2.csv
  - benchmark_scalability_m2.csv

Done. Compare split_eval_sec here vs M1 CSV for speedup figures.


In [16]:
# Create Synthetic-only
!cp milestone2_code.cu milestone2_synthetic_fixed.cu

# Keep only Synthetic
!sed -i '/std::vector<DatasetSpec> datasets = {/,/};/c\        std::vector<DatasetSpec> datasets = {\n            {"Synthetic_dataset_1000k_200f", "synthetic_1000k_200f.csv", true}\n        };' milestone2_synthetic_fixed.cu

# Add save
!sed -i '/print_result_summary(res);/a \            save_metrics_csv("benchmark_metrics_m2.csv", metrics);\n            save_scalability_csv("benchmark_scalability_m2.csv", scalability_rows);' milestone2_synthetic_fixed.cu

# Compile and run (will take 5-15 min)
!nvcc -O2 -std=c++17 -arch=sm_75 milestone2_synthetic_fixed.cu -o milestone2_synthetic_fixed
!./milestone2_synthetic_fixed

   MILESTONE 2 — Final Integrated CPU-GPU Tree Trainer

[Test: pure node -> leaf]
  [PASS]

[Test: level-wise toy split]
  [PASS]

[Loading] Synthetic_dataset_1000k_200f from: synthetic_1000k_200f.csv

[Dataset: Synthetic_dataset_1000k_200f | Mode: GPU (Hybrid CPU-GPU)]
  samples/features/classes : 1000000 / 200 / 10
  train/test samples       : 800000 / 200000
  total_train_sec          : 12.561294
  data_prep_sec            : 0.007409  (Step 2: index packing)
  split_eval_sec           : 1.121338  (Step 3: GPU kernels — compare vs M1)
  apply_split_sec          : 0.529750  (Step 4: CPU apply splits)
  predict_sec              : 0.115847
  train_acc                : 97.052375%
  test_acc                 : 96.949500%

[Artifacts saved]
  - benchmark_metrics_m2.csv
  - benchmark_scalability_m2.csv

Done. Compare split_eval_sec here vs M1 CSV for speedup figures.


In [18]:
from google.colab import files
files.download('benchmark_metrics_m2.csv')
files.download('benchmark_scalability_m2.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
# Run scalability for each dataset and save to unique files

# 1. IRIS
!./milestone2_iris_fixed
!cp benchmark_scalability_m2.csv iris_scalability.csv
print("✅ Iris scalability saved")

# 2. SHUTTLE
!./milestone2_shuttle_fixed
!cp benchmark_scalability_m2.csv shuttle_scalability.csv
print("✅ Shuttle scalability saved")

# 3. LETTER
!./milestone2_letter_fixed
!cp benchmark_scalability_m2.csv letter_scalability.csv
print("✅ Letter scalability saved")

# 4. SKIN
!./milestone2_skin_fixed
!cp benchmark_scalability_m2.csv skin_scalability.csv
print("✅ Skin scalability saved")

# 5. SYNTHETIC (already have it)
!cp benchmark_scalability_m2.csv synthetic_scalability.csv
print("✅ Synthetic scalability saved")

print("\n✅ All scalability files saved!")

   MILESTONE 2 — Final Integrated CPU-GPU Tree Trainer

[Test: pure node -> leaf]
  [PASS]

[Test: level-wise toy split]
  [PASS]

[Loading] Iris from: Iris.csv

[Dataset: Iris | Mode: GPU (Hybrid CPU-GPU)]
  samples/features/classes : 150 / 5 / 3
  train/test samples       : 120 / 30
  total_train_sec          : 0.000440
  data_prep_sec            : 0.000001  (Step 2: index packing)
  split_eval_sec           : 0.000240  (Step 3: GPU kernels — compare vs M1)
  apply_split_sec          : 0.000004  (Step 4: CPU apply splits)
  predict_sec              : 0.000001
  train_acc                : 100.000000%
  test_acc                 : 100.000000%
✅ Iris scalability saved
   MILESTONE 2 — Final Integrated CPU-GPU Tree Trainer

[Test: pure node -> leaf]
  [PASS]

[Test: level-wise toy split]
  [PASS]

[Loading] Shuttle from: shuttle.csv

[Dataset: Shuttle | Mode: GPU (Hybrid CPU-GPU)]
  samples/features/classes : 14500 / 9 / 7
  train/test samples       : 11600 / 2900
  total_train_sec       

In [20]:
from google.colab import files

for f in ['iris_scalability.csv', 'shuttle_scalability.csv', 'letter_scalability.csv', 'skin_scalability.csv', 'synthetic_scalability.csv']:
    files.download(f)
    print(f"Downloaded {f}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded iris_scalability.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Downloaded skin_scalability.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded synthetic_scalability.csv


In [21]:
# Create complete benchmark_metrics_m2.csv with all 5 datasets
content = '''dataset,n_samples,n_features,n_classes,train_samples,test_samples,total_train_sec,data_prep_sec,split_eval_sec,apply_split_sec,predict_sec,train_accuracy,test_accuracy
Iris,150,5,3,120,30,0.000405,0.000001,0.000224,0.000004,0.000002,1.000000,1.000000
Shuttle,14500,9,7,11600,2900,0.005504,0.000064,0.001822,0.001039,0.000202,0.977759,0.982069
LetterRecognition,20000,16,26,16000,4000,0.013309,0.000094,0.002825,0.001666,0.000352,0.601125,0.585750
Skin_NonSkin,245057,3,2,196045,49012,0.040398,0.000789,0.005438,0.015838,0.002967,0.992165,0.992777
Synthetic_dataset_1000k_200f,1000000,200,10,800000,200000,12.561294,0.007409,1.121338,0.529750,0.115847,0.970524,0.969495'''

with open('benchmark_metrics_m2.csv', 'w') as f:
    f.write(content)

print("✅ File created!")

# Download it
from google.colab import files
files.download('benchmark_metrics_m2.csv')

✅ File created!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>